# Pneuma-Seeker — Quick Start

This notebook shows a concise example of configuring Pneuma-Seeker for a sample dataset. It complements the README by demonstrating the exact configuration and a sample query so you can reproduce the workflow.

> Prerequisite: Install and run the backend and frontend (see the `installation` section in [README.md](README.md#installation)). Ensure you can access the backend at [http://127.0.0.1:8000/docs](http://127.0.0.1:8000/docs) and the frontend at [http://127.0.0.1:8080](http://127.0.0.1:8080).

# Step 1: Index the dataset

We use the `archeology` dataset from KramaBench (available in this repository under `data_src/archeology`). The dataset is a collection of CSV files, so we use the CSV connector to read and index the data. The example below posts an indexing request to the backend `/index` endpoint.

Ensure that the `.env` file configuration is correct. In particular, the embedding model should be the same for both indexing and querying. Relevant variables include `LLM_PATH` and `EMBED_MODEL_PATH`. Refer to [model_factory.py](src/pneuma_seeker/services/language_model/model_factory.py) for the exact logic on how `Pneuma-Seeker` interfaces with these models through `LMService`.

In [ ]:
import json
from pathlib import Path

import requests

In [ ]:
BASE_DIR = Path.cwd().resolve()
DATASET_PATH = (BASE_DIR / "data_src/archeology/dataset").resolve()
METADATA_PATH = (BASE_DIR / "data_src/archeology/metadata.csv").resolve()

payload = {
    "dataset_name": "archeology",
    "connector_config": {
        "directory_path": str(DATASET_PATH),
        "metadata_path": str(METADATA_PATH),
        "type": "csv",
    },
    "overwrite": False,  # Change to True if you want to overwrite existing index
    "schema_summaries": None,
}

response = requests.post("http://localhost:8000/index", json=payload)
print(response.status_code)
print(json.dumps(response.json(), indent=2))

Indexing runs asynchronously. You can monitor progress in [src/pneuma_seeker/main.out](src/pneuma_seeker/main.out) and call the `/index/[dataset_name]/latest` endpoint. When indexing completes, proceed to Step 2.

In [ ]:
response = requests.get("http://localhost:8000/index/archeology/latest")
print(response.status_code)
print(json.dumps(response.json(), indent=2))

The indexing process can take around 4 minutes when using OpenAI's o3 model. We originally designed a dynamic batch size selection algorithm to accelerate this step, but it is currently disabled due to limitations in API-based models: OpenAI enforces request- and token-level rate limits, which prevents effective GPU-style batching. Refer to this GitHub issue: https://github.com/TheDataStation/pneuma-seeker/issues/19

# Step 2: Configure the backend

Follow the instructions in [README.md](README.md#installation): copy `.env.example` to `.env` and fill the required values. At minimum, set the following variables according to your environment:

```bash
OPENAI_API_KEY="<your OpenAI API key>"
LLM_PATH="<OpenAI model for LLM inference, e.g. o4-mini>"
```

Alternatively, you can use local models (`qwen3.5` and `qwen3-embedding`) through Ollama (see https://ollama.com/download) by replacing `OPENAI_API_KEY` with:
```bash
LLM_PATH=ollama
EMBED_MODEL_PATH=ollama
```

# Step 3: Query Pneuma-Seeker

Open the frontend at [http://localhost:3000](http://localhost:3000). Login using the admin account (notice `ADMIN_EMAIL` and `ADMIN_PASSWORD` in `.env`). Select "archeology" as the dataset and ask questions!

For example: "What city that is located in both the southern and western hemispheres has the highest population?"

![OpenWebUI Chat](docs/figures/pneuma_seeker_ui.png)